# Electrical Finishing Data Analysis and Model Preparation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv("electrical_finishing_final_reclassified.csv")
df.head()

## 3. Initial Inspection

In [ ]:
df.info()
df.describe(include="all")
df.nunique()

## 4. Data Quality Checks

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df[df["Price_EGP"] <= 0]

## 5. Data Cleaning

In [ ]:
df_clean = df.copy()

text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["Price_EGP"]).reset_index(drop=True)

df_clean.head()

## 6. Quality Level Analysis

In [ ]:
df_clean.groupby("Quality_Level")["Price_EGP"].describe()

In [ ]:
quality_price = (
    df_clean.groupby("Quality_Level", as_index=False)["Price_EGP"]
    .mean()
)

quality_price["Quality_Level"] = pd.Categorical(
    quality_price["Quality_Level"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

quality_price = quality_price.sort_values("Quality_Level")

sns.barplot(data=quality_price, x="Quality_Level", y="Price_EGP")
plt.title("Average Price by Quality Level")
plt.show()

## 7. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean["Price_EGP"], bins=30, kde=True)
plt.title("Price Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_clean,
    x="Quality_Level",
    y="Price_EGP",
    order=["Low", "Medium", "High"]
)
plt.title("Price Distribution by Quality Level")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_clean,
    x="Subcategory",
    y="Price_EGP",
    estimator="mean",
    order=df_clean.groupby("Subcategory")["Price_EGP"].mean().sort_values(ascending=False).index
)
plt.xticks(rotation=45)
plt.title("Average Price by Subcategory")
plt.show()

## 8. Outlier Analysis

In [ ]:
Q1 = df_clean["Price_EGP"].quantile(0.25)
Q3 = df_clean["Price_EGP"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean["Price_EGP"] < lower_bound) |
    (df_clean["Price_EGP"] > upper_bound)
]

outliers.head()

## 9. Feature Engineering

In [ ]:
df_model = df_clean.copy()

df_model["Estimated_Quantity"] = np.where(
    df_model["Quantity_Rule"] == "Per_Area",
    np.ceil(df_model["Rule_Value"] * 100),
    df_model["Rule_Value"]
)

df_model["Estimated_Total_Cost"] = (
    df_model["Estimated_Quantity"] * df_model["Price_EGP"]
)

df_model.head()

## 10. Apartment Requirement Logic

In [ ]:
def calculate_electrical_requirements(
    apartment_area,
    rooms,
    bathrooms=1
):
    return {
        "Apartment_Area_m2": apartment_area,
        "Rooms": rooms,
        "Bathrooms": bathrooms
    }

## 11. Cost Estimation

In [ ]:
def estimate_electrical_cost(
    data,
    apartment_area,
    quality_level=None
):
    filtered = data.copy()

    if quality_level is not None:
        filtered = filtered[
            filtered["Quality_Level"] == quality_level
        ]

    estimated_quantity = np.where(
        filtered["Quantity_Rule"] == "Per_Area",
        np.ceil(filtered["Rule_Value"] * apartment_area),
        filtered["Rule_Value"]
    )

    return (estimated_quantity * filtered["Price_EGP"]).sum()

## 12. Budget-Based Recommendation

In [ ]:
def recommend_electrical_products(
    data,
    budget,
    quality_level,
    application=None
):
    filtered = data[
        data["Quality_Level"] == quality_level
    ].copy()

    if application is not None:
        filtered = filtered[
            filtered["Application"].astype(str).str.contains(
                str(application),
                case=False,
                na=False
            )
        ]

    filtered = filtered[
        filtered["Price_EGP"] <= budget
    ]

    return filtered.sort_values("Price_EGP")

## 13. Multi-File Model Integration Structure

In [ ]:
df_model["Finishing_Category"] = "Electrical"

common_columns = [
    "Finishing_Category",
    "Category",
    "Subcategory",
    "Product_Name",
    "Brand",
    "Quality_Level",
    "Price_EGP",
    "Unit",
    "Quantity_Rule",
    "Rule_Value",
    "Required_For",
    "Optional"
]

electrical_for_master_model = df_model[
    [col for col in common_columns if col in df_model.columns]
].copy()

electrical_for_master_model.head()

## 14. Prepare Data for Future Master Model

In [ ]:
electrical_model_data = electrical_for_master_model.copy()

electrical_model_data = electrical_model_data.dropna(
    subset=["Price_EGP"]
).reset_index(drop=True)

electrical_model_data.info()

## 15. Baseline Price Prediction Model

In [ ]:
df_ml = df_model.copy()

candidate_features = [
    col for col in df_ml.columns
    if col not in [
        "Price_EGP",
        "Quality_Level",
        "Quality_Price_Band",
        "Estimated_Total_Cost"
    ]
]

X = df_ml[candidate_features]
y = df_ml["Price_EGP"]

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

## 16. Train and Evaluate Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

baseline_model.fit(X_train, y_train)

predictions = baseline_model.predict(X_test)

model_results = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
    "R2": r2_score(y_test, predictions)
}

model_results

## 17. Actual vs Predicted

In [ ]:
comparison = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": predictions
})

comparison.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=comparison,
    x="Actual_Price",
    y="Predicted_Price"
)
plt.title("Actual vs Predicted Price")
plt.show()

## 18. Final Validation and Export

In [ ]:
model_ready_data = df_model.copy()

model_ready_data.to_csv(
    "electrical_finishing_model_ready_reclassified.csv",
    index=False
)

model_ready_data.head()

## 19. Final Project Role

هذه البيانات ستكون جزءًا من الـ Master Product Data الخاص بمشروع حساب تكلفة تشطيب شقة كاملة.

سيتم دمج بيانات الكهرباء مع باقي أقسام التشطيب مع الاحتفاظ بالتفاصيل الخاصة بكل قسم.

النظام النهائي سيعتمد على:
- مساحة الشقة
- عدد الغرف
- عدد الحمامات
- الميزانية
- مستوى التشطيب
- اختيارات المستخدم

وفي النهاية سيحسب:
- الكميات المطلوبة
- تكلفة كل قسم
- المنتجات المناسبة
- البدائل داخل الميزانية
- إجمالي تكلفة التشطيب

وسيتم استخدام البيانات لاحقًا في إنشاء Apartment Scenarios وتجهيز Training Dataset للموديل النهائي باستخدام scikit-learn.